In [1]:
import os
from pathlib import Path
# from openai import OpenAI
from sentence_transformers import SentenceTransformer
from chromadb import PersistentClient
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

In [ ]:
load_dotenv(override=True)

# openai = OpenAI()
# EMBEDDING_MODEL = "text-embedding-3-small"

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
chroma = PersistentClient(path="./chroma_db")
try:
    chroma.delete_collection("transcripts")
except:
    pass
collection = chroma.get_or_create_collection("transcripts")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
def load_documents(base_path="transcripts"):
    """
    Loops through the folders from the base_path and reads the txt files inside them
    Reads each transcript and stores text + metadata in a list of dicts
    Returns one dict(documents) per file with text, week, day, source
    """
    documents = [] #List of dicts
    base = Path(base_path) #Converts base_path into a Path object to use .glob and .iterdir()
    

# Load documents from the transcripts folder
# If its not a folder, continue
# If it is a folder, iterate through all files in the folder
    for week_folder in sorted(base.iterdir()):
        if not week_folder.is_dir():
            continue
    # Iterate through all files in the week folder using .glob
        for file in sorted(week_folder.glob("*.txt")):
            text = file.read_text(encoding="utf-8")
            # Create a dictionary for each document and add it to the list
            documents.append({
                "text": text, # text of the transcript
                "week": week_folder.name, # name of the week folder
                "day": file.stem, # name of the file
                "source": str(file) # path of the file. file is a Path object and its converted to a string
            })
            print(f"Loaded: {file} ({len(text)} characters)")

    print(f"\nTotal documents loaded: {len(documents)}")
    return documents

documents = load_documents() #documents is the result of load_documents()

Loaded: transcripts/week5/day1.txt (9721 characters)

Total documents loaded: 1


In [ ]:
def chunk_documents(documents):
    """
    Creates the chunks using RecursiveCharacterTextSplitter
    Adds the chunks to a list
    Returns the list of chunks
    """
    # Creates the chunks using Langchains RecursiveCharacterTextSplitter
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        separators=["--- Lecture", "\n\n", "\n", " "]
    )
    # Loops on the documents list which is a list of dicts from the cell above
    chunks = []
    for doc in documents:
        pieces = splitter.split_text(doc["text"]) #split_text splits documents into chunks based on the parameters from RecursiveCharacterTextSplitter.
        
        # Loop on the chunk and then append this dict into the chunks list
        for piece in pieces: 
            chunks.append({
                "text": piece, # text of the chunk
                "week": doc["week"], # name of the folder
                "day": doc["day"], # day of the folder
                "source": doc["source"] # path of the file. file is a Path object
            })

    print(f"Total chunks created: {len(chunks)}")
    print(f"\n--- Sample chunk ---\n")
    print(chunks[0]["text"])
    return chunks # A list of dicts that include text, week, day, and source

In [23]:
def embed_and_store(chunks):
    """
    Create texts, metadatas, and ids and stores it in variables
    Encodes the texts into vectors
    Adds the vector into the chroma database
    """
    texts = [chunk["text"] for chunk in chunks] # A list of all the text in chunks
    metadatas = [{"week": chunk["week"], "day": chunk["day"], "source": chunk["source"]} for chunk in chunks] #Creates metadata for the week, day, and the source.
    ids = [f"{chunk['source']}_{i}" for i, chunk in enumerate(chunks)] # Names each chunk with the file name and an index

    embeddings = embedder.encode(texts, show_progress_bar=True).tolist() # Encodes the texts into vector embeddings using the sentence transformer model and converts to a list

    # Adds the vectors into the chroma database with the following data
    collection.add(
        documents=texts,
        embeddings=embeddings,
        metadatas=metadatas,
        ids=ids
    )
    print(f"Stored {len(chunks)} chunks in Chroma.")

chunks = chunk_documents(documents) # A list of dicts that include text, week, day, and source saved into the variable chunks
embed_and_store(chunks) # Run this function on the list of dicts that are in chunks

Total chunks created: 22

--- Sample chunk ---

Well, what a moment.
You've made it.
You've made it to the second half of the course and you've made it to rag week.
Week five.
Let's just quickly review what you can do.
You can already code with frontier models.
You can build AI assistants with tools.
You can work with open source models.
You can confidently choose the right model for your project, which you can back up with facts, with
benchmarks, with metrics.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Stored 22 chunks in Chroma.


In [ ]:
def retrieve(query, n_results=3):
    """
    Returns the top n_results chunks most similar to the query
    """
    
    query_embedding = embedder.encode(query).tolist()  # Converts the query into a vector and puts it in a list

    # Uses the list of vectors from the query and gives n_results of similar vectors through a dictionary
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )

    return results["documents"][0], results["metadatas"][0] # filtered n_results of the similar vectors to the query.

docs, metas = retrieve("What is RAG?") # Saves the result of retrieve to docs and metas
for doc, meta in zip(docs, metas): # Zips them together as a tuple
    print(f"Source: {meta['source']}\n{doc}\n---") # Print the source, and then the text of the chunk

Source: transcripts/week5/day1.txt
information in there in the prompt.
That is the basic motivation behind Rag.
Nothing more complicated than that.
And you probably thought of that already.
So I like to describe this as the small idea behind rag.
And once we've done this, we'll then talk about the big idea behind the small idea behind Rag, which
is essentially what I just said is, okay, just imagine this.
So we've got this set up in the middle there where it says code.
---
Source: transcripts/week5/day1.txt
It's obvious this is the small idea behind Rag.
Again, you probably already got this.
So what I'm going to do now is I want to make this small idea come to life for you, even though most
of you probably already completely get it.
But let's just go through a little exercise, a small example of the small idea.
So what we're going to do is we're going to imagine that we are working for an insurance tech startup.
This is going to be something in the insurance tech industry.
---
Source: 